# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a complete guide for loading, exploring, and analyzing the [FAIR\u005e2] dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()
print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview

Examine available record sets and their fields. We will list their `@id` values for reference throughout this notebook.

In [ ]:
# Get the list of available record sets
record_sets = dataset.metadata.record_sets
print('Available record sets (by @id):')
for rs in record_sets:
    print(f"@id: {rs.id}, name: {rs.name}")

# For demonstration, print detailed info for the first record set
if record_sets:
    first_record_set = record_sets[0]
    print(f"\nFields and columns for record set '{first_record_set.id}':")
    for field in first_record_set.fields:
        print(f"  Field @id: {field.id}, name: {field.name}, dataType: {field.data_type}")
        for column in field.columns:
            print(f"    Column @id: {column.id}, name: {column.name}, source: {column.source}")

## 3. Data Extraction

Load all data from the available record set(s) into DataFrame(s) for analysis. Each record set is identified by its unique `@id`.

In [ ]:
# Extract data into DataFrames
dataframes = {}
record_set_ids = [rs.id for rs in record_sets]

for record_set_id in record_set_ids:
    # Retrieve records from each record set
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"\n{record_set_id} columns:", df.columns.tolist())
    print(df.head())

## 4. Exploratory Data Analysis (EDA)

Process and analyze the data. This may include filtering for high protein abundance, normalizing fields like molecular weight, and grouping by descriptive attributes.

Replace the field `@id` values below with those relevant to your dataset as listed in the data overview section.

In [ ]:
# --- EDA Example: Filter and Normalize a Numeric Field ---
# Choose the first record set for demonstration, adjust as needed
if record_set_ids:
    record_set_id = record_set_ids[0]
    df = dataframes[record_set_id]
    print(f"Working with record set: {record_set_id}")

    # Find numeric fields for demonstration (float/int)
    numeric_cols = df.select_dtypes(include=['int', 'float']).columns.tolist()
    print("Numeric fields:", numeric_cols)

    # Example: Use the first numeric field for filtering/normalization
    if numeric_cols:
        numeric_field = numeric_cols[0]

        threshold = df[numeric_field].mean() if not pd.isnull(df[numeric_field].mean()) else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records where {numeric_field} > {threshold:.2f}:")
        print(filtered_df.head())

        # Normalize this numeric field
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by another field if available and numeric (e.g. description or accession)
        candidate_group_fields = [c for c in df.columns if c != numeric_field and df[c].nunique() < 10 and df[c].dtype == 'object']
        if candidate_group_fields:
            group_field = candidate_group_fields[0]
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped by {group_field} (mean {numeric_field}):")
            print(grouped_df.head())
        else:
            print("No suitable categorical field found for grouping.")
    else:
        print("No numeric fields found for EDA.")

## 5. Visualization

Visualize distributions or relationships between fields in the dataset. Adjust field names as needed based on your data overview.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example visualization: Histogram of the selected numeric field
if record_set_ids and numeric_cols:
    df = dataframes[record_set_id]
    numeric_field = numeric_cols[0]
    plt.figure(figsize=(8, 5))
    sns.histplot(df[numeric_field].dropna(), bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()

# Pairplot if more than 1 numeric field
if record_set_ids and len(numeric_cols) > 1:
    sns.pairplot(df[numeric_cols].dropna())
    plt.show()

## 6. Conclusion

- Used `mlcroissant` to access metadata and data programmatically.\n- Explored available record sets and field structure (all referenced by `@id`).\n- Loaded data into DataFrames for flexible analysis.\n- Performed basic EDA: filtered data, normalized numeric fields, and optionally grouped by attributes.\n- Generated basic visualizations to explore distributions and relationships.\n
Next steps could include applying advanced analysis, feature engineering, or training machine learning models based on the processed data.